# Bayesian Sensitivity Estimate Example

This example shows how to calculate the **Bayesian sensitivity** for a given signal process using the `SNSensitivityEstimate` package.

Where the first example (`ex1`) only looked at the *expected background counts*, here we go a step further: we build a full Bayesian model of signal + background, fit it to a simulated ("pseudo-") dataset, and from the posterior we derive a **sensitivity** — the half-life that the experiment could realistically probe given the expected background level, exposure, and signal efficiency.

In [1]:
import Pkg;
Pkg.activate("/sps/nemo/scratch/mpetro/Sensitivity_training/")
using SNSensitivityEstimate, UnROOT, FHist, CairoMakie, CSV, DataFramesMeta
using Random, LinearAlgebra, Statistics, Distributions, BAT, BinnedModels, StatsBase, DensityInterface, IntervalSets, SpecialFunctions, ValueShapes
using Distributions

  Activating project at `/sps/nemo/scratch/mpetro/Sensitivity_training`
[ Info: Precompiling SNSensitivityEstimate [4337d2dd-53cf-40a5-bc43-6e7cc5722bd2]
┌ Warning: Module PrecompileTools with build ID fafbfcfd-e880-2b99-000f-58856e04c9e8 is missing from the cache.
│ This may mean PrecompileTools [aea7be01-6a6a-4083-8856-8a6e6704d82a] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948
[ Info: Skipping precompilation since __precompile__(false). Importing SNSensitivityEstimate [4337d2dd-53cf-40a5-bc43-6e7cc5722bd2].
[ Info: Precompiling DataFramesMeta [1313f7d8-7da2-5740-9ea0-a2ca25f37964]
┌ Warning: Module PrecompileTools with build ID fafbfcfd-e880-2b99-000f-58856e04c9e8 is missing from the cache.
│ This may mean PrecompileTools [aea7be01-6a6a-4083-8856-8a6e6704d82a] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:1948
[ Info: Skipping precompilation since __precompile__(false). Importing DataFrame

## Step 1: Load the data

The loading step is identical to `ex1_background.jl`: we read the `data_info.csv` index, open the corresponding ROOT files with `UnROOT`, and wrap each tree as a `LazyTree`.

In [6]:
data_info = CSV.read("/sps/nemo/scratch/mpetro/Sensitivity_training/data/data_info.csv", DataFrame)
file_paths = joinpath.("/sps/nemo/scratch/mpetro/Sensitivity_training/data", data_info.file_name)
data_files = [UnROOT.ROOTFile(file_path) for file_path in file_paths]
tree_name = "tree"
variables = keys(data_files[1][tree_name])
data_tables = [UnROOT.LazyTree(data_file, tree_name, variables) for data_file in data_files]

5-element Vector{LazyTree with 26 branches:
sameSide, reconstructedEnergy2, Pext, avgE, z2Reconstructed...
}:
 ┌─────┬──────────┬─────────────────┬────────────┬───────────┬───────────────────
│ Row │ sameSide │ reconstructedEn │ Pext       │ avgE      │ z2Reconstructed  ⋯
│     │ Float32  │ Float32         │ Float32    │ Float32   │ Float32          ⋯
├─────┼──────────┼─────────────────┼────────────┼───────────┼───────────────────
│ 1   │ 0.0      │ 417.84256       │ 2.062762e- │ 539.3591  │ 36.14118         ⋯
│ 2   │ 1.0      │ 66.81933        │ 0.4553107  │ 872.25037 │ -62.85587        ⋯
│ 3   │ 1.0      │ 286.42862       │ 7.6636536e │ 1108.881  │ 796.15796        ⋯
│ 4   │ 0.0      │ 163.08542       │ 0.00024017 │ 701.16925 │ -1109.882        ⋯
│ 5   │ 0.0      │ 420.899         │ 5.03516e-1 │ 474.4948  │ -773.49896       ⋯
│ 6   │ 0.0      │ 771.45776       │ 3.876098e- │ 1004.4069 │ 591.2683         ⋯
│ 7   │ 0.0      │ 1874.1747       │ 1.8641002e │ 1229.9344 │ 57.884754        

### Step 1.1: Apply data cuts

In `ex1` we did not impose any cuts on the data. For a sensitivity estimate to be meaningful, we first need to restrict the data to a region where signal-like events are enriched relative to background — this is the same selection logic used in any counting-experiment analysis.

We'll cut on:
- `sumE` — the total reconstructed energy of the event,
- `dy`, `dz` — vertex distance in y and z direction (in mm),
- `Pint`, `Pext` — internal/external probability 

First we define which variables we want to cut on:

In [ ]:
var_names = [ 
    "sumE", 
    "dy", 
    "dz",
    "Pint",
    "Pext",
    ]

Next we define the bounds of each cut. Here we use a simple selection: `sumE` > 300 keV, `dy` < 100 mm, `dz` < 100 mm, `Pint` > 0.01, `Pext` < 0.01 — i.e. events with enough energy, a vertex match, and a topology favoring an internal-decay hypothesis.

In [ ]:
var_bounds = (
    sumE = (300, 3500),
    dy = (0, 100),
    dz = (0, 100),
    Pint = (0.01, 1),
    Pext = (0, 0.01),
)

Now we apply the cuts with `SNSensitivityEstimate.filter_data`, which filters each table according to `var_names` and `var_bounds` before we load it into a `DataProcess`:

In [ ]:
filtered_data_tables = [SNSensitivityEstimate.filter_data(table, var_names, var_bounds) for table in data_tables]

As before, we wrap each filtered table into a `SNSensitivityEstimate.DataProcess` object — this time using the cut data instead of the raw data:

In [ ]:
processes = [
    SNSensitivityEstimate.DataProcess(
        collect(filtered_data_tables[i].sumE), # Here we specify a single variable to be used. In later stages we will use N-dimensional data vectors.
        String(data_info.process[i]),
        data_info.is_signal[i],
        data_info.activity[i],
        data_info.time_s[i],
        data_info.n_sim[i],
        0:100:3500,
        data_info.amount[i],
    ) for i in 1:length(filtered_data_tables)
]

> **Optional exercise:** redo the `ex1_background.jl` example with these cuts applied (or try your own cuts) and see how the expected background spectrum changes.

## Step 2: Build normalized signal and background histograms

The Bayesian fit doesn't operate on raw event counts directly — it needs **shape templates** (normalized histograms) for signal and background, plus a **pseudo-dataset** to fit them to. We build all three in this step.

First, we separate out the signal process (`bb0nu_foil_bulk`, the neutrinoless double-beta decay signal) and obtain its histogram:

In [ ]:
signal = SNSensitivityEstimate.get_process("bb0nu_foil_bulk", processes)[1]
signal_hist = SNSensitivityEstimate.get_bkg_counts_1D(signal)

Then we collect all the remaining (background) processes and get their histograms too:

In [ ]:
bkg_names = [ p.isotopeName for p in processes if !p.signal ]
background = [SNSensitivityEstimate.get_process(name, processes)[1] for name in bkg_names]
bkg_hist = [SNSensitivityEstimate.get_bkg_counts_1D(p) for p in background]

For the fit, the templates need to be **shape-only** (i.e. probability densities, not raw expected counts), so we normalize each histogram by its bin widths:

In [ ]:
bkg_hist_normed = FHist.normalize.(bkg_hist, width = true)
signal_hist_normed = FHist.normalize(signal_hist, width = true)

In [ ]:
sample_hists = [SNSensitivityEstimate.get_pseudo_spectrum(b) for b in bkg_hist] 
# merge them together into a single pseudo-experiment
data_hist = merge(sample_hists...)

Let's compare the pseudo-data against the underlying background model, to sanity-check that the sampling looks reasonable:

In [ ]:
let
    f = CairoMakie.Figure(size = (800, 600))
    a = CairoMakie.Axis(f[1, 1], title = "Pseudo-data vs background model", xlabel = "sum of energies", ylabel = "counts")
    CairoMakie.stephist!(a, sum(bkg_hist), label = "background model", color = :red)
    CairoMakie.stephist!(a, data_hist, label = "pseudo-data", color = :blue)
    CairoMakie.axislegend(a, position = :lt)
    f
end

## Step 3: Define the Bayesian model

With our pseudo-data and templates ready, we now set up the Bayesian inference problem. In Bayesian statistics this means specifying three ingredients: a **prior** (what we assume before seeing the data), a **likelihood** (how probable the data are given the model parameters), and the resulting **posterior** (what we believe about the parameters after seeing the data).

### Step 3.1: Prior

We use an *uninformed* (flat) prior for each process's relative activity — i.e. we don't favor any particular value ahead of time. `As` is the signal amplitude, `Ab` is a vector of amplitudes, one per background process:

In [ ]:
prior = ValueShapes.NamedTupleDist(
    As = Distributions.Uniform(1e-20, 1), # one prior for signal - flat prior between 0 and 1
    Ab = [Distributions.Uniform(1e-20,1) for _ in 1:length(bkg_hist)] # same with background priors
)   

### Step 3.2: Likelihood

The likelihood is built from the normalized signal/background histograms. It is defined in section 5.4.3 of the accompanying PhD thesis (will be uploaded to docDB when available). Conceptually: the fit parameters are the *relative weights* of each process's contribution to the total pseudo-data histogram —

- `A_s` — relative amount of signal,
- `A_b` — vector of relative amounts of each background process —

and the likelihood tells us how probable the observed `data_hist` is for a given choice of these weights.

In [ ]:
likelihood = SNSensitivityEstimate.make_hist_likelihood_uniform(
    data_hist,
    signal_hist_normed,
    bkg_hist_normed
)

### Step 3.3: Posterior

`BAT.jl` provides a convenient interface to combine prior and likelihood into a posterior: we simply call `BAT.PosteriorMeasure` with both as arguments.

In [ ]:
posterior = BAT.PosteriorMeasure(likelihood, prior)

### Step 3.4: MCMC sampler setup

To actually explore the posterior distribution, we use Markov Chain Monte Carlo (MCMC) sampling, as implemented in `BAT.jl` (full documentation at [bat.github.io/BAT.jl/stable](https://bat.github.io/BAT.jl/stable/)). Here we just configure some sensible defaults:

- `burnin` — settings for the *burn-in* phase, where the chain is allowed to wander before we start collecting samples, so it can "forget" its (possibly poor) starting point,
- `mcmcalgo` — the sampling algorithm itself; we use a simple random-walk Metropolis sampler.

In [ ]:
burnin = BAT.MCMCMultiCycleBurnin(max_ncycles = 30, nsteps_final=1000)
mcmcalgo = BAT.RandomWalk()

## Step 4: Run the MCMC sampling

We now actually run the sampler. `nsteps` is the number of MCMC steps per chain, and `nchains` is the number of independent chains run in parallel (running multiple chains lets us check that they all converge to the same posterior, as a consistency check).

> This step is computationally the heaviest part of the notebook and may take a little while to run.

In [ ]:
samples, _ = BAT.bat_sample( posterior, BAT.MCMCSampling(mcalg = mcmcalgo, burnin = burnin, nsteps = 10_000, nchains = 4))

We can print some basic summary statistics of the posterior samples — the mode (most likely value), mean, and standard deviation for each fitted parameter:

In [ ]:
println("Mode: $(mode(samples))")
println("Mean: $(mean(samples))")
println("Stddev: $(std(samples))")

In [ ]:
binned_unshaped_samples, _ = BAT.bat_transform(Vector, samples)

Now we can look at the posterior distribution of each individual parameter — i.e. how tightly constrained the signal amplitude and each background amplitude are by the fit — by histogramming the MCMC samples for each one:

In [ ]:
let
    f = CairoMakie.Figure(size = (800, 1000))
    a1 = CairoMakie.Axis(f[1, 1], title = "Posterior distribution of signal", xlabel = "signal ratio", ylabel = "counts")
    d = [par[1] for par in binned_unshaped_samples.v]
    h = FHist.Hist1D(d; binedges= range(0, maximum(d), length = 100))
    CairoMakie.stephist!(a1, h, label = "A_s", color = :blue)
    CairoMakie.axislegend(a1, position = :ct)

    labels = data_info.process[.!data_info.is_signal]
    colors = Makie.wong_colors()
    for i in 1:length(bkg_hist)
        a2 = CairoMakie.Axis(f[i+1, 1], title = "Posterior distribution of $(labels[i])", xlabel = "background ratio", ylabel = "counts")       
        d = [par[i+1] for par in binned_unshaped_samples.v]
        h = FHist.Hist1D(d; binedges= range(0, maximum(d), length = 100))
        CairoMakie.stephist!(a2, h, label = "A_b$(i)", color = colors[i])
        CairoMakie.axislegend(a2, position = :ct)
    end
    save("data/out/ex2/posterior_distributions.png", f, px_per_unit = 2)
    f
end

## Step 5: Convert the posterior into a sensitivity

We now turn the posterior distribution of the signal amplitude into a physical **sensitivity** — i.e. the half-life that this (pseudo-)experiment is sensitive to.

The convention used here is the standard "90% credible interval" sensitivity: we take the **90% quantile** of the posterior distribution of the signal amplitude as the upper limit on the number of signal events the data are consistent with, given no significant excess was found. This is then converted into a half-life via:

$$T_{1/2} = \ln(2) \cdot \frac{N_A \, m \, t \, \varepsilon}{W \cdot \mu_{S}^{90}}$$

where:
- $N_A$ is Avogadro's number,
- $m$ is the mass of the isotope of interest,
- $t$ is the live-time of the experiment,
- $\varepsilon$ is the signal detection efficiency in the chosen region of interest (ROI),
- $W$ is the molar mass of the isotope,
- $\mu_S^{90}$ is the 90% credible-interval upper limit on the expected number of signal events, obtained from the posterior.

In [ ]:
nDataPoints = integral(data_hist)
muS = [par[1] for par in binned_unshaped_samples.v]
exp_mu_signal_90 = BAT.smallest_credible_intervals(muS, nsigma_equivalent=1.65)[1].right * nDataPoints # this gives the smallest credible interval

Na = SNparams["Nₐ"]
m = SNparams["foilMass"] * SNparams["a"]
t = SNparams["tYear"]
W = SNparams["W"]

ROI_a, ROI_b = 300, 3500-50
eff = lookup(signal, ROI_a, ROI_b) 
t12 = log(2) * (Na * m * t * eff / W) / exp_mu_signal_90

In [ ]:
println("Sensitivity (90% CI) in terms of half-life: $(t12) years)")
println("for a signal efficiency of $(eff)")
println("in the ROI of $(ROI_a) to $(ROI_b) keV.")
println("equivalent to an expected number of signal events of $(round(exp_mu_signal_90, digits= 3))")

## (Optional) Step 6: Visualize the fit result

As a sanity check, let's plot the fitted spectrum — built from the posterior mean amplitudes of each process — against the pseudo-background, to see how well the fit reproduces the input shapes.

In [ ]:
amps = mean(samples) # get the amplitudes of each process from the posterior samples, which are the mean of the samples for each parameter.

In [ ]:
let 
    colors = Makie.wong_colors()
    bkg_amps = amps[2] ./ sum(amps[2]) # normalize the background amplitudes to sum to 1
    labels = data_info.process[.!data_info.is_signal]
    f = Figure(size = (1200,800), fontsize = 26)
    a = Axis(f[1,1], xlabel = "sum E", ylabel = "Counts / 17.5kg.yr", title = "Fitted spectrum of a single pseudo-experiment", limits = (300,3500, 0, nothing) )
    h_total = sum(bkg_hist)
    n_total = integral(h_total)
    CairoMakie.stephist!(a, h_total, color = :black, label = "Total pseudo-background", linewidth = 4)
    
    h_fit = [ a*normalize(h, width = false)*n_total for (h, a) in zip(bkg_hist, bkg_amps)]

    for i in 1:length(h_fit[1:end])
        CairoMakie.stephist!(a, h_fit[i], color = colors[i], label = labels[i], linewidth = 4)
    end
    CairoMakie.hist!(a, sum(h_fit), label = "Fitted total", color = (colors[1], 0.4), )
    
    axislegend(a; position = :rt, patchsize = (45,35), labelsize = 20)
    save("data/out/ex2/fit_result.png", f, px_per_unit = 2)
    f
end

## (For later) Step 7: From a single pseudo-experiment to a robust sensitivity

The procedure above gives the sensitivity for a **single** pseudo-experiment. But MCMC sampling — and the Poisson-sampling of the pseudo-data itself — is a *stochastic* process: running it again with a different random seed will give a somewhat different answer. A sensitivity quoted from just one pseudo-experiment is therefore not representative of the true sensitivity of the experiment.

To obtain a robust number suitable for publication, we repeat the *entire* pipeline — generate new pseudo-data, run a fresh MCMC fit, compute a new sensitivity — many times, and report the **median** of the resulting sensitivity distribution. This whole loop is encapsulated by `SNSensitivityEstimate.get_sens_bayes_uniform`, which performs one full pseudo-experiment (sampling + fit + sensitivity calculation) per call.

> This will take a while, since each iteration involves a full MCMC run — `n_max` is kept small (10) here for demonstration purposes; in practice you would use a much larger number of pseudo-experiments to get a stable median.

In [ ]:
t_halfs = []
n_max = 10 # number of pseudo-experiments to run
for i in 1:n_max

    try 
        println("Running pseudo-experiment $(i) of $(n_max)")
        sens = SNSensitivityEstimate.get_sens_bayes_uniform(
            bkg_hist, 
            signal, 
            prior; 
            ROI_a = ROI_a, ROI_b = ROI_b, nsteps = 5*10^4, nchains = 4)
        println(sens)
        push!(t_halfs, sens)
    catch
        @warn "failed fit" 
        continue
    end
end

Finally, let's plot the distribution of sensitivities obtained across all pseudo-experiments, and mark the median value — this median is the number we would actually quote as "the" sensitivity of the experiment.

In [ ]:
let
    f = CairoMakie.Figure(size = (800, 600))
    a = CairoMakie.Axis(f[1, 1], title = "Distribution of sensitivities from $(n_max) pseudo-experiments", xlabel = "sensitivity (years)", ylabel = "counts")
    h = FHist.Hist1D(t_halfs; binedges= range(0, maximum(t_halfs), length = n_max))
    CairoMakie.stephist!(a, h, label = "sensitivity distribution", color = :blue)

    median_sens = StatsBase.median(t_halfs)
    CairoMakie.vlines!(a, [median_sens], color = :red, label = "median sensitivity: $(round(median_sens, sigdigits= 3)) years")
    CairoMakie.axislegend(a, position = :ct)
    save("data/out/ex2/sensitivity_distribution.png", f, px_per_unit = 2)
    f
end